# TP1

## Generacion de la clase_ternaria

In [ ]:
require( "data.table" )

# leo el dataset
dataset <- fread("/content/datasets/competencia_01_crudo.csv" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]



Loading required package: data.table



In [ ]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202101,BAJA+1,622
202101,BAJA+2,825
202101,CONTINUA,160080
202102,BAJA+1,831
202102,BAJA+2,1032
202102,CONTINUA,160292
202103,BAJA+1,1039
202103,BAJA+2,951
202103,CONTINUA,161119


ESTABILIZACIÓN POR IPC

In [ ]:
# =================== ESTABILIZACIÓN POR IPC (base 202104 = 100) ===================

library(data.table)

# 1) Tabla IPC ( % ya convertidos a índice con base abril=100)
ipc <- data.table(
  foto_mes = c(202101L, 202102L, 202103L, 202104L, 202105L, 202106L),
  indice   = c(90.7,     92.1,     95.4,     100.0,    103.3,    106.6)
)

# 2) Merge por foto_mes
stopifnot("foto_mes" %in% names(dataset))
dataset <- merge(dataset, ipc, by = "foto_mes", all.x = TRUE, sort = FALSE)

# 3) Deflactor relativo a 202104
dataset[, defl_factor := indice / 100]


# 4) Detectar columnas de montos a deflactar
#    - Montos "generales": empiezan con 'm'
#    - Montos de tarjetas: nombres como 'Master_m...' o 'Visa_m...'
montos_m_prefix <- grep("^m", names(dataset), value = TRUE)
montos_card     <- grep("^(Master|Visa)_m", names(dataset), value = TRUE)

montos_all <- unique(c(montos_m_prefix, montos_card))

# por si alguna coincide pero no es numérica
montos_all <- montos_all[vapply(dataset[, ..montos_all], is.numeric, logical(1))]

# 5) Deflactar IN-PLACE: x := x / defl_factor  (expresado en pesos constantes 202104)
for (cn in montos_all) {
  dataset[, (cn) := get(cn) / pmax(defl_factor, 1e-9)]
}

# 6) (Opcional) dejar rastro
#dataset[, ipc_base := 202104L]


cat("IPC aplicado (base 202104). Columnas deflactadas:", length(montos_all), "\n")
# =================== /ESTABILIZACIÓN POR IPC ===================



✅ IPC aplicado (base 202104). Columnas deflactadas: 73 


In [ ]:
# eliminar columnas de IPC para no laguearlas ni usarlas como features
drop_ipc <- intersect(c("indice","defl_factor","ipc_base"), names(dataset))
if (length(drop_ipc)) dataset[, (drop_ipc) := NULL]

In [ ]:
ncol(dataset)
colnames(dataset)

[1] 155

[1] "foto_mes"                            
  [2] "numero_de_cliente"                   
  [3] "active_quarter"                      
  [4] "cliente_vip"                         
  [5] "internet"                            
  [6] "cliente_edad"                        
  [7] "cliente_antiguedad"                  
  [8] "mrentabilidad"                       
  [9] "mrentabilidad_annual"                
 [10] "mcomisiones"                         
 [11] "mactivos_margen"                     
 [12] "mpasivos_margen"                     
 [13] "cproductos"                          
 [14] "tcuentas"                            
 [15] "ccuenta_corriente"                   
 [16] "mcuenta_corriente_adicional"         
 [17] "mcuenta_corriente"                   
 [18] "ccaja_ahorro"                        
 [19] "mcaja_ahorro"                        
 [20] "mcaja_ahorro_adicional"              
 [21] "mcaja_ahorro_dolares"                
 [22] "cdescubierto_preacordado"            
 [23] "mcuentas_saldo"                      
 [24] "ctarjeta_debito"                     
 [25] "ctarjeta_debito_transacciones"       
 [26] "mautoservicio"                       
 [27] "ctarjeta_visa"                       
 [28] "ctarjeta_visa_transacciones"         
 [29] "mtarjeta_visa_consumo"               
 [30] "ctarjeta_master"                     
 [31] "ctarjeta_master_transacciones"       
 [32] "mtarjeta_master_consumo"             
 [33] "cprestamos_personales"               
 [34] "mprestamos_personales"               
 [35] "cprestamos_prendarios"               
 [36] "mprestamos_prendarios"               
 [37] "cprestamos_hipotecarios"             
 [38] "mprestamos_hipotecarios"             
 [39] "cplazo_fijo"                         
 [40] "mplazo_fijo_dolares"                 
 [41] "mplazo_fijo_pesos"                   
 [42] "cinversion1"                         
 [43] "minversion1_pesos"                   
 [44] "minversion1_dolares"                 
 [45] "cinversion2"                         
 [46] "minversion2"                         
 [47] "cseguro_vida"                        
 [48] "cseguro_auto"                        
 [49] "cseguro_vivienda"                    
 [50] "cseguro_accidentes_personales"       
 [51] "ccaja_seguridad"                     
 [52] "cpayroll_trx"                        
 [53] "mpayroll"                            
 [54] "mpayroll2"                           
 [55] "cpayroll2_trx"                       
 [56] "ccuenta_debitos_automaticos"         
 [57] "mcuenta_debitos_automaticos"         
 [58] "ctarjeta_visa_debitos_automaticos"   
 [59] "mttarjeta_visa_debitos_automaticos"  
 [60] "ctarjeta_master_debitos_automaticos" 
 [61] "mttarjeta_master_debitos_automaticos"
 [62] "cpagodeservicios"                    
 [63] "mpagodeservicios"                    
 [64] "cpagomiscuentas"                     
 [65] "mpagomiscuentas"                     
 [66] "ccajeros_propios_descuentos"         
 [67] "mcajeros_propios_descuentos"         
 [68] "ctarjeta_visa_descuentos"            
 [69] "mtarjeta_visa_descuentos"            
 [70] "ctarjeta_master_descuentos"          
 [71] "mtarjeta_master_descuentos"          
 [72] "ccomisiones_mantenimiento"           
 [73] "mcomisiones_mantenimiento"           
 [74] "ccomisiones_otras"                   
 [75] "mcomisiones_otras"                   
 [76] "cforex"                              
 [77] "cforex_buy"                          
 [78] "mforex_buy"                          
 [79] "cforex_sell"                         
 [80] "mforex_sell"                         
 [81] "ctransferencias_recibidas"           
 [82] "mtransferencias_recibidas"           
 [83] "ctransferencias_emitidas"            
 [84] "mtransferencias_emitidas"            
 [85] "cextraccion_autoservicio"            
 [86] "mextraccion_autoservicio"            
 [87] "ccheques_depositados"                
 [88] "mcheques_depositados"                
 [89] "ccheques_emitidos"                 

SAC

In [ ]:
# ====== AJUSTE DE AGUINALDO  ======

evento_aguinaldo <- 202106L
tau <- 0.45

# señales de haberes
dataset[, payroll_monto := fcoalesce(mpayroll, 0) + fcoalesce(mpayroll2, 0)]
dataset[, payroll_freq  := fcoalesce(cpayroll_trx, 0L) + fcoalesce(cpayroll2_trx, 0L)]
dataset[, has_payroll   := as.integer(payroll_monto > 0 | payroll_freq > 0)]

# lags para sueldos
dataset[, `:=`(
  mpayroll_lag1       = shift(mpayroll, 1L),
  mpayroll2_lag1      = shift(mpayroll2, 1L),
  pay_monto_lag1      = shift(payroll_monto, 1L),
  cpayroll_trx_lag1   = shift(cpayroll_trx, 1L),
  cpayroll2_trx_lag1  = shift(cpayroll2_trx, 1L)
), by = numero_de_cliente]

# rolling max 6m PREVIO (excluye el mes actual)
dataset[, pay_max6_prev := frollapply(shift(payroll_monto, 1L),
                                      6L, max, align = "right", na.rm = TRUE),
        by = numero_de_cliente]

# SAC teórico en base a histórico previo
dataset[, sac_teorico := 0.5 * pay_max6_prev]

# salto vs mayo
dataset[, inc_vs_may := payroll_monto - pay_monto_lag1]

# detectar SAC solo si hay haberes y el salto es “≈ media remuneración previa”
dataset[, sac_flag := (foto_mes == evento_aguinaldo) &
                      ((fcoalesce(mpayroll, 0) + fcoalesce(mpayroll2, 0)) > 0) &
                      is.finite(inc_vs_may) & is.finite(pay_max6_prev) &
                      (inc_vs_may >= tau * pay_max6_prev)]

# 1) Ajuste mpayroll / mpayroll2 proporcional sin crear columnas extra
dataset[sac_flag == TRUE, `:=`(
  mpayroll  = {
    total <- pmax(mpayroll + mpayroll2, 1e-9)
    desc  <- pmin(sac_teorico, payroll_monto)
    pmax(0, mpayroll  - desc * (mpayroll  / total))
  },
  mpayroll2 = {
    total <- pmax(mpayroll + mpayroll2, 1e-9)
    desc  <- pmin(sac_teorico, payroll_monto)
    pmax(0, mpayroll2 - desc * (mpayroll2 / total))
  }
)]

# 2) Ajuste de saldo (usa lag/roll existentes, pero no crea columnas nuevas)
# cuantiles del TRAIN para cap
idx_train <- dataset$foto_mes %in% c(202101L, 202102L, 202103L, 202104L)
q1_s  <- as.numeric(quantile(dataset[idx_train, mcuentas_saldo], 0.01, na.rm = TRUE, type = 7))
q99_s <- as.numeric(quantile(dataset[idx_train, mcuentas_saldo], 0.99, na.rm = TRUE, type = 7))


if (!"mcuentas_saldo_lag1" %in% names(dataset))
  dataset[, mcuentas_saldo_lag1 := shift(mcuentas_saldo, 1L), by = numero_de_cliente]
if (!"mcuentas_saldo_roll_mean3" %in% names(dataset))
  dataset[, mcuentas_saldo_roll_mean3 := frollmean(mcuentas_saldo, 3L, align = "right"), by = numero_de_cliente]

dataset[sac_flag == TRUE & foto_mes == evento_aguinaldo,
        mcuentas_saldo := {
          base <- fifelse(!is.na(mcuentas_saldo_lag1), mcuentas_saldo_lag1, mcuentas_saldo_roll_mean3)
          inc  <- pmax(0, mcuentas_saldo - base)
          desc <- pmin(sac_teorico, payroll_monto)
          nuevo <- mcuentas_saldo - pmin(inc, desc)
          pmin(q99_s, pmax(q1_s, nuevo))
        }]


# Limpieza mínima de auxiliares de este bloque
dataset[, `:=`(inc_vs_may = NULL, sac_teorico = NULL)]
# ====== /SAC ======

  # --- limpiar auxiliares del bloque SAC ---
cols_temp <- c(
  "payroll_monto","payroll_freq","has_payroll","pay_monto_lag1",
  "pay_max6","sac_teorico","inc_vs_may","sac_flag","mcuentas_saldo_roll_mean3","pay_max6_prev"
)
cols_temp <- intersect(cols_temp, names(dataset))
if (length(cols_temp)) dataset[, (cols_temp) := NULL]

aux <- intersect(c("mpayroll_lag1","mpayroll2_lag1","pay_monto_lag1",
                   "cpayroll_trx_lag1","cpayroll2_trx_lag1",
                   "mcuentas_saldo_lag1","mcuentas_saldo_roll_mean3",
                   "pay_max6_prev","inc_vs_may","sac_teorico","sac_flag"),
                 names(dataset))
if (length(aux)) dataset[, (aux) := NULL]

In [ ]:
ncol(dataset)
colnames(dataset)

[1] 155

[1] "foto_mes"                            
  [2] "numero_de_cliente"                   
  [3] "active_quarter"                      
  [4] "cliente_vip"                         
  [5] "internet"                            
  [6] "cliente_edad"                        
  [7] "cliente_antiguedad"                  
  [8] "mrentabilidad"                       
  [9] "mrentabilidad_annual"                
 [10] "mcomisiones"                         
 [11] "mactivos_margen"                     
 [12] "mpasivos_margen"                     
 [13] "cproductos"                          
 [14] "tcuentas"                            
 [15] "ccuenta_corriente"                   
 [16] "mcuenta_corriente_adicional"         
 [17] "mcuenta_corriente"                   
 [18] "ccaja_ahorro"                        
 [19] "mcaja_ahorro"                        
 [20] "mcaja_ahorro_adicional"              
 [21] "mcaja_ahorro_dolares"                
 [22] "cdescubierto_preacordado"            
 [23] "mcuentas_saldo"                      
 [24] "ctarjeta_debito"                     
 [25] "ctarjeta_debito_transacciones"       
 [26] "mautoservicio"                       
 [27] "ctarjeta_visa"                       
 [28] "ctarjeta_visa_transacciones"         
 [29] "mtarjeta_visa_consumo"               
 [30] "ctarjeta_master"                     
 [31] "ctarjeta_master_transacciones"       
 [32] "mtarjeta_master_consumo"             
 [33] "cprestamos_personales"               
 [34] "mprestamos_personales"               
 [35] "cprestamos_prendarios"               
 [36] "mprestamos_prendarios"               
 [37] "cprestamos_hipotecarios"             
 [38] "mprestamos_hipotecarios"             
 [39] "cplazo_fijo"                         
 [40] "mplazo_fijo_dolares"                 
 [41] "mplazo_fijo_pesos"                   
 [42] "cinversion1"                         
 [43] "minversion1_pesos"                   
 [44] "minversion1_dolares"                 
 [45] "cinversion2"                         
 [46] "minversion2"                         
 [47] "cseguro_vida"                        
 [48] "cseguro_auto"                        
 [49] "cseguro_vivienda"                    
 [50] "cseguro_accidentes_personales"       
 [51] "ccaja_seguridad"                     
 [52] "cpayroll_trx"                        
 [53] "mpayroll"                            
 [54] "mpayroll2"                           
 [55] "cpayroll2_trx"                       
 [56] "ccuenta_debitos_automaticos"         
 [57] "mcuenta_debitos_automaticos"         
 [58] "ctarjeta_visa_debitos_automaticos"   
 [59] "mttarjeta_visa_debitos_automaticos"  
 [60] "ctarjeta_master_debitos_automaticos" 
 [61] "mttarjeta_master_debitos_automaticos"
 [62] "cpagodeservicios"                    
 [63] "mpagodeservicios"                    
 [64] "cpagomiscuentas"                     
 [65] "mpagomiscuentas"                     
 [66] "ccajeros_propios_descuentos"         
 [67] "mcajeros_propios_descuentos"         
 [68] "ctarjeta_visa_descuentos"            
 [69] "mtarjeta_visa_descuentos"            
 [70] "ctarjeta_master_descuentos"          
 [71] "mtarjeta_master_descuentos"          
 [72] "ccomisiones_mantenimiento"           
 [73] "mcomisiones_mantenimiento"           
 [74] "ccomisiones_otras"                   
 [75] "mcomisiones_otras"                   
 [76] "cforex"                              
 [77] "cforex_buy"                          
 [78] "mforex_buy"                          
 [79] "cforex_sell"                         
 [80] "mforex_sell"                         
 [81] "ctransferencias_recibidas"           
 [82] "mtransferencias_recibidas"           
 [83] "ctransferencias_emitidas"            
 [84] "mtransferencias_emitidas"            
 [85] "cextraccion_autoservicio"            
 [86] "mextraccion_autoservicio"            
 [87] "ccheques_depositados"                
 [88] "mcheques_depositados"                
 [89] "ccheques_emitidos"                 

LAGS

In [ ]:
library(data.table)
setorder(dataset, numero_de_cliente, foto_mes)
setDTthreads(0L)




# Feature Engineering Historico
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente","foto_mes","clase_ternaria","clase01",
                  "training","azar","cliente_antiguedad","cliente_edad",
                  "Master_status","Visa_status")
) )

#Prealocar (ahora que ya sabemos cuántas lagueables hay)
alloc.col(dataset, ncol(dataset) + 4L * length(cols_lagueables) + 50L)



dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags de orden 1
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}





# Limpiar auxiliares (una sola vez)
dataset[, c("mes_id","mes_id_lag1","mes_id_lag2","cons1","cons2") := NULL]
gc()

# sanity check
faltan_lag2 <- setdiff(paste0(cols_lagueables, "_lag2"), names(dataset))
if (length(faltan_lag2)) stop("No se crearon estas _lag2: ", paste(faltan_lag2, collapse=", "))

cat(" LAGS/DELTAS OK — lag1:", length(cols_lagueables),
    " lag2:", length(cols_lagueables),
    " delta1:", length(cols_lagueables),
    " delta2:", length(cols_lagueables), "\n")

foto_mes,numero_de_cliente,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,⋯,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo,clase_ternaria
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<int>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<chr>
202101,249221323,1,0,0,46,93,3385.0496,17300.00,1409.1621,⋯,4,0.000,-22969.647,0.00000,2814,7434.8181,6,0,10540.187,CONTINUA
202102,249221323,1,0,0,46,94,3676.2758,20742.30,1336.0152,⋯,-3,55085.168,-55085.168,0.00000,2842,8377.0467,8,0,8049.251,CONTINUA
202103,249221323,1,0,0,46,95,5598.7212,25178.42,792.2956,⋯,1,0.000,-53179.706,0.00000,2873,11219.0776,7,0,0.000,CONTINUA
202104,249221323,1,0,0,46,96,5356.6200,29189.81,1470.2000,⋯,2,0.000,0.000,0.00000,2903,8094.7300,6,0,0.000,CONTINUA
202105,249221323,1,0,0,46,97,2409.7289,30499.30,976.5731,⋯,5,0.000,-16516.989,0.00000,2934,8762.8074,6,0,2043.950,NA
202106,249221323,1,0,0,46,98,2099.7280,30120.82,1151.2852,⋯,0,0.000,-18777.617,0.00000,2964,15104.7655,7,0,2321.792,NA
202101,249227600,1,0,0,42,278,547.0893,16892.83,461.9294,⋯,11,14627.883,-10653.264,0.00000,5541,10385.7773,8,0,2095.105,CONTINUA
202102,249227600,1,0,0,42,279,785.9935,15910.45,298.3931,⋯,4,0.000,-14405.527,0.00000,5569,12972.3453,9,0,2419.870,CONTINUA
202103,249227600,1,0,0,42,280,825.9958,14466.54,478.4696,⋯,7,0.000,-18429.067,0.00000,5600,7237.1803,5,0,2200.912,CONTINUA


Warning message in `[.data.table`(dataset, , `:=`(c("mes_id", "mes_id_lag1", "mes_id_lag2", :
“Tried to assign NULL to column 'mes_id', but this column does not exist to remove”
Warning message in `[.data.table`(dataset, , `:=`(c("mes_id", "mes_id_lag1", "mes_id_lag2", :
“Tried to assign NULL to column 'mes_id_lag1', but this column does not exist to remove”
Warning message in `[.data.table`(dataset, , `:=`(c("mes_id", "mes_id_lag1", "mes_id_lag2", :
“Tried to assign NULL to column 'mes_id_lag2', but this column does not exist to remove”
Warning message in `[.data.table`(dataset, , `:=`(c("mes_id", "mes_id_lag1", "mes_id_lag2", :
“Tried to assign NULL to column 'cons1', but this column does not exist to remove”
Warning message in `[.data.table`(dataset, , `:=`(c("mes_id", "mes_id_lag1", "mes_id_lag2", :
“Tried to assign NULL to column 'cons2', but this column does not exist to remove”


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,742529,39.7,1454657,77.7,1454657,77.7
Vcells,551315774,4206.3,914777998,6979.3,551653961,4208.8


✅ LAGS/DELTAS OK — lag1: 148  lag2: 148  delta1: 148  delta2: 148 


In [ ]:
ncol(dataset)
colnames(dataset)

[1] 747

[1] "foto_mes"                                   
  [2] "numero_de_cliente"                          
  [3] "active_quarter"                             
  [4] "cliente_vip"                                
  [5] "internet"                                   
  [6] "cliente_edad"                               
  [7] "cliente_antiguedad"                         
  [8] "mrentabilidad"                              
  [9] "mrentabilidad_annual"                       
 [10] "mcomisiones"                                
 [11] "mactivos_margen"                            
 [12] "mpasivos_margen"                            
 [13] "cproductos"                                 
 [14] "tcuentas"                                   
 [15] "ccuenta_corriente"                          
 [16] "mcuenta_corriente_adicional"                
 [17] "mcuenta_corriente"                          
 [18] "ccaja_ahorro"                               
 [19] "mcaja_ahorro"                               
 [20] "mcaja_ahorro_adicional"                     
 [21] "mcaja_ahorro_dolares"                       
 [22] "cdescubierto_preacordado"                   
 [23] "mcuentas_saldo"                             
 [24] "ctarjeta_debito"                            
 [25] "ctarjeta_debito_transacciones"              
 [26] "mautoservicio"                              
 [27] "ctarjeta_visa"                              
 [28] "ctarjeta_visa_transacciones"                
 [29] "mtarjeta_visa_consumo"                      
 [30] "ctarjeta_master"                            
 [31] "ctarjeta_master_transacciones"              
 [32] "mtarjeta_master_consumo"                    
 [33] "cprestamos_personales"                      
 [34] "mprestamos_personales"                      
 [35] "cprestamos_prendarios"                      
 [36] "mprestamos_prendarios"                      
 [37] "cprestamos_hipotecarios"                    
 [38] "mprestamos_hipotecarios"                    
 [39] "cplazo_fijo"                                
 [40] "mplazo_fijo_dolares"                        
 [41] "mplazo_fijo_pesos"                          
 [42] "cinversion1"                                
 [43] "minversion1_pesos"                          
 [44] "minversion1_dolares"                        
 [45] "cinversion2"                                
 [46] "minversion2"                                
 [47] "cseguro_vida"                               
 [48] "cseguro_auto"                               
 [49] "cseguro_vivienda"                           
 [50] "cseguro_accidentes_personales"              
 [51] "ccaja_seguridad"                            
 [52] "cpayroll_trx"                               
 [53] "mpayroll"                                   
 [54] "mpayroll2"                                  
 [55] "cpayroll2_trx"                              
 [56] "ccuenta_debitos_automaticos"                
 [57] "mcuenta_debitos_automaticos"                
 [58] "ctarjeta_visa_debitos_automaticos"          
 [59] "mttarjeta_visa_debitos_automaticos"         
 [60] "ctarjeta_master_debitos_automaticos"        
 [61] "mttarjeta_master_debitos_automaticos"       
 [62] "cpagodeservicios"                           
 [63] "mpagodeservicios"                           
 [64] "cpagomiscuentas"                            
 [65] "mpagomiscuentas"                            
 [66] "ccajeros_propios_descuentos"                
 [67] "mcajeros_propios_descuentos"                
 [68] "ctarjeta_visa_descuentos"                   
 [69] "mtarjeta_visa_descuentos"                   
 [70] "ctarjeta_master_descuentos"                 
 [71] "mtarjeta_master_descuentos"                 
 [72] "ccomisiones_mantenimiento"                  
 [73] "mcomisiones_mantenimiento"                  
 [74] "ccomisiones_otras"                          
 [75] "mcomisiones_otras"                          
 [76] "cforex"                                     
 [77] "cforex_buy"                                

In [ ]:
fwrite( dataset,
    file =  "/content/datasets/competencia_01.csv.gz",
    sep = ","
)

### 2.2 Optimizacion Hiperparámetros

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Oct 12 00:10:34 2025"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,743978,39.8,1454657,77.7,1454657,77.7
Vcells,1413991,10.8,731822399,5583.4,552236412,4213.3


### 2.2.2 Carga de Librerias

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

if( !require("DiceKriging") ) install.packages("DiceKriging")
require("DiceKriging")

if( !require("mlrMBO") ) install.packages("mlrMBO")
require("mlrMBO")

Loading required package: parallel

Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, use, warnings


Loading required package: primes

Loading required package: rlist

Loading required packa

### 2.2.3 Definicion de Parametros

In [ ]:
PARAM <- list()
PARAM$experimento <- "TP1_11_4"
PARAM$semilla_primigenia <- 102191


In [ ]:
# training y future
# BO: 202103 completo + 202101, 202102 solo BAJAS
PARAM$train_full    <- c(202103L)
PARAM$train_posonly <- c(202101L, 202102L)

PARAM$train <- c(202101, 202102,202103)
PARAM$valid <- c(202104)
PARAM$train_final <- c(202101, 202102, 202103, 202104)
PARAM$future <- c(202106)
PARAM$semilla_kaggle <- 314159
PARAM$cortes <- seq(6000, 19000, by= 250)

In [ ]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 1.0

In [ ]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 1200,
  learning_rate= 0.02,
  feature_fraction= 0.5,
  num_leaves= 750,
  min_data_in_leaf= 5000
)


In [ ]:
# Aqui se cargan los bordes de los hiperparametros de la BO
PARAM$hyperparametertuning$hs <- makeParamSet(
  makeIntegerParam("num_iterations", lower= 8L, upper= 2048L),
  makeNumericParam("learning_rate", lower= 0.01, upper= 0.3),
  makeNumericParam("feature_fraction", lower= 0.1, upper= 1.0),
  makeIntegerParam("num_leaves", lower= 8L, upper= 2048L),
  makeIntegerParam("min_data_in_leaf", lower= 1L, upper= 8000L),
  #makeNumericParam("min_gain_to_split", lower=0.0, upper=5.0),
  makeNumericParam("lambda_l1", lower=0.0, upper=50.0),
  makeNumericParam("lambda_l2", lower=0.0, upper=50.0)

)

In [ ]:
PARAM$hyperparametertuning$iteraciones <- 50 # iteraciones bayesianas

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [ ]:
# iniciliazo el dataset de realidad, para medir ganancia
realidad_inicializar <- function( pfuture, pparam) {

  # datos para verificar la ganancia
  drealidad <- pfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]

  particionar(drealidad,
    division= c(3, 7),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_kaggle
  )

  return( drealidad )
}

In [ ]:
# evaluo ganancia en los datos de la realidad

realidad_evaluar <- function( prealidad, pprediccion) {

  prealidad[ pprediccion,
    on= c("numero_de_cliente", "foto_mes"),
    predicted:= i.Predicted
  ]

  tbl <- prealidad[, list("qty"=.N), list(fold, predicted, clase_ternaria)]

  res <- list()
  res$public  <- tbl[fold==1 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 780000, -20000))]/0.3
  res$private <- tbl[fold==2 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 780000, -20000))]/0.7
  res$total <- tbl[predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 780000, -20000))]

  prealidad[, predicted:=NULL]
  return( res )
}

### 2.2.4  Preprocesamiento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/competencia_01.csv.gz", stringsAsFactors= TRUE)

In [ ]:
idx_train <- (dataset$foto_mes %in% c(PARAM$train_full, PARAM$train_posonly))
dataset_train <- dataset[idx_train]

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [ ]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]


dataset_train[
  (foto_mes %in% PARAM$train_full) |
  (foto_mes %in% PARAM$train_posonly &  clase01==1L),
  training := 1L
]

In [ ]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("numero_de_cliente", "clase_ternaria", "clase01", "azar", "training")
)


In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 166419

[1] 745

2.2.5 Configuracion Bayesian Optimization

In [ ]:
# En el argumento x llegan los parmaetros de la bayesiana
#  devuelve la AUC en cross validation del modelo entrenado

EstimarGanancia_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)
  nrounds_es <- if (!is.null(param_completo$num_iterations)) as.integer(param_completo$num_iterations) else 1200L
  es_rounds  <- max(50L, round(nrounds_es * 0.15))
  param_completo$data_random_seed <- NULL

  # entreno LightGBM
  modelocv <- lgb.cv(
    data= dtrain,
    nfold= PARAM$hyperparametertuning$xval_folds,
    stratified= TRUE,
    param= param_completo,
    nrounds = nrounds_es,
    early_stopping_rounds = es_rounds,
    verbose = -1
  )

  # obtengo la ganancia
  AUC <- modelocv$best_score

  # hago espacio en la memoria
  rm(modelocv)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y"), " AUC ", AUC)

  return(AUC)
}


In [ ]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- EstimarGanancia_AUC_lightgbm # la funcion que voy a maximizar

configureMlr(show.learner.output= FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo

obj.fun <- makeSingleObjectiveFunction(
  fn= funcion_optimizar, # la funcion que voy a maximizar
  minimize= FALSE, # estoy Maximizando la ganancia
  noisy= TRUE,
  par.set= PARAM$hyperparametertuning$hs, # definido al comienzo del programa
  has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
  save.on.disk.at.time= 600, # se graba cada 600 segundos
  save.file.path= kbayesiana
) # se graba cada 600 segundos

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
  ctrl,
  iters= PARAM$hyperparametertuning$iteraciones
) # cantidad de iteraciones

# defino el método estandar para la creacion de los puntos iniciales,
# los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit= makeMBOInfillCritEI())

# establezco la funcion que busca el maximo
surr.km <- makeLearner(
  "regr.km",
  predict.type= "se",
  covtype= "matern3_2",
  control= list(trace= TRUE)
)


2.2.6 Corrida Bayesian Optimization

In [ ]:
# inicio la optimizacion bayesiana, retomando si ya existe
# es la celda mas lenta de todo el notebook

if (!file.exists(kbayesiana)) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue(kbayesiana) # retomo en caso que ya exista
}

Computing y column(s) for design. Not provided.

Sun Oct 12 00:12:22 2025 AUC 0.969267768569759

Sun Oct 12 00:14:06 2025 AUC 0.971513928172447

Sun Oct 12 00:14:37 2025 AUC 0.969875745487902

Sun Oct 12 00:16:10 2025 AUC 0.970965720081404

Sun Oct 12 00:17:17 2025 AUC 0.970540638988469

Sun Oct 12 00:18:43 2025 AUC 0.971692466545159

Sun Oct 12 00:20:32 2025 AUC 0.97148403296034

Sun Oct 12 00:22:45 2025 AUC 0.972031725043802

Sun Oct 12 00:23:50 2025 AUC 0.972219845479454

Sun Oct 12 00:25:23 2025 AUC 0.970001415292683

Sun Oct 12 00:30:28 2025 AUC 0.97037587703066

Sun Oct 12 00:32:25 2025 AUC 0.970862366944886

Sun Oct 12 00:33:09 2025 AUC 0.972843843967818

Sun Oct 12 00:33:46 2025 AUC 0.970740326256383

Sun Oct 12 00:34:28 2025 AUC 0.971451267639534

Sun Oct 12 00:38:14 2025 AUC 0.973638645172364

Sun Oct 12 00:38:28 2025 AUC 0.955642263191653

Sun Oct 12 00:40:06 2025 AUC 0.971488608915034

Sun Oct 12 00:40:35 2025 AUC 0.970610184998523

Sun Oct 12 00:42:54 2025 AUC 0.9711395167

In [ ]:

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)
colnames( tb_bayesiana)

[1] "num_iterations"   "learning_rate"    "feature_fraction" "num_leaves"      
 [5] "min_data_in_leaf" "lambda_l1"        "lambda_l2"        "y"               
 [9] "dob"              "eol"              "error.message"    "exec.time"       
[13] "ei"               "error.model"      "train.time"       "prop.type"       
[17] "propose.time"     "se"               "mean"

In [ ]:
# almaceno los resultados de la Bayesian Optimization
# y capturo los mejores hiperparametros encontrados

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)

tb_bayesiana[, iter := .I]

# ordeno en forma descendente por AUC = y
setorder(tb_bayesiana, -y)

# grabo para eventualmente poder utilizarlos en OTRA corrida
fwrite( tb_bayesiana,
  file= "BO_log.txt",
  sep= "\t"
)

# los mejores hiperparámetros son los que quedaron en el registro 1 de la tabla
PARAM$out$lgbm$mejores_hiperparametros <- tb_bayesiana[
  1, # el primero es el de mejor AUC
  setdiff(colnames(tb_bayesiana),
    c("y","dob","eol","error.message","exec.time","ei","error.model",
      "train.time","prop.type","propose.time","se","mean","iter")),
  with= FALSE
]


PARAM$out$lgbm$y <- tb_bayesiana[1, y]


In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
print(PARAM$out$lgbm$mejores_hiperparametros)
print(PARAM$out$lgbm$y)

   num_iterations learning_rate feature_fraction num_leaves min_data_in_leaf
            <int>         <num>            <num>      <int>            <int>
1:            994    0.04483601        0.1651085       2047              832
   lambda_l1 lambda_l2
       <num>     <num>
1:  6.771127  5.828662
[1] 0.9738073


## 2.3  VALID

In [ ]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
dataset_train <- dataset[foto_mes %in% PARAM$train]
dataset_train[,.N,clase_ternaria]

clase_ternaria,N
<fct>,<int>
CONTINUA,481491
BAJA+2,2808
BAJA+1,2492


In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain_valid <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)



#### Hyperparameters

In [ ]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 102191

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 6.771127

$lambda_l2
[1] 5.828662

$max_bin
[1] 31

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 994

$learning_rate
[1] 0.04483601

$feature_fraction
[1] 0.1651085

$num_leaves
[1] 2047

$min_data_in_leaf
[1] 832

In [ ]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [ ]:
# Entrenar en TRAIN y predecir en VALID

modelo_valid <- lgb.train(
  data= dtrain_valid,
  param= param_normalizado
)


In [ ]:
# Conjunto de validación
valid_set <- dataset[ foto_mes %in% PARAM$valid ]

pred_valid <- predict(
  modelo_valid,
  data.matrix(valid_set[, campos_buenos, with = FALSE])
)

In [ ]:
# 4) Tabla valid con probas y ganancia por fila
tb_valid <- valid_set[, .(numero_de_cliente, clase_ternaria)]
tb_valid[, prob := pred_valid]
setorder(tb_valid, -prob)

In [ ]:
# Ganancia por “invitar” fila a fila (ordenada por prob desc)
tb_valid[, gain_row := fifelse(clase_ternaria=="BAJA+2", 780000, -20000)]
tb_valid[, gain_cum := cumsum(gain_row)]

In [ ]:
# 5) Imprimir TODOS los cortes K y elegir el mejor
K_grid <- PARAM$cortes                     # ej. seq(6000, 19000, by=250)
k_eff  <- pmin(K_grid, nrow(tb_valid))
res_k  <- data.table(K = K_grid, Ganancia = tb_valid$gain_cum[k_eff])

print(res_k)                               # <<--- todos los K con su ganancia

        K  Ganancia
    <num>     <num>
 1:  6000 344000000
 2:  6250 346200000
 3:  6500 348400000
 4:  6750 356200000
 5:  7000 356800000
 6:  7250 358200000
 7:  7500 360400000
 8:  7750 363400000
 9:  8000 363200000
10:  8250 364600000
11:  8500 362800000
12:  8750 364200000
13:  9000 368000000
14:  9250 369400000
15:  9500 370000000
16:  9750 365800000
17: 10000 367200000
18: 10250 367000000
19: 10500 370000000
20: 10750 372200000
21: 11000 372000000
22: 11250 375800000
23: 11500 374800000
24: 11750 373000000
25: 12000 370400000
26: 12250 367800000
27: 12500 366000000
28: 12750 365800000
29: 13000 364000000
30: 13250 362200000
31: 13500 361200000
32: 13750 359400000
33: 14000 354400000
34: 14250 353400000
35: 14500 351600000
36: 14750 349000000
37: 15000 349600000
38: 15250 347000000
39: 15500 344400000
40: 15750 341000000
41: 16000 338400000
42: 16250 336600000
43: 16500 334800000
44: 16750 330600000
45: 17000 331200000
46: 17250 329400000
47: 17500 326000000
48: 17750 325000000


In [ ]:
best_idx  <- which.max(res_k$Ganancia)
K_offline <- res_k$K[best_idx]
G_offline <- res_k$Ganancia[best_idx]
cat(">> K_offline (valid) =", K_offline, "| ganancia =", G_offline, "\n")
# (opcional) guardar tabla K vs ganancia
fwrite(res_k, file = "ganancia_por_K_valid.csv")



>> K_offline (valid) = 11250 | ganancia = 375800000 


In [ ]:
# 6) Importancia y guardado del modelo de valid (por prolijidad)
tb_importancia <- as.data.table(lgb.importance(modelo_valid))
fwrite(tb_importancia, file = "impo_valid.txt", sep = "\t")
lgb.save(modelo_valid, "modelo_valid.txt")

In [ ]:
write_yaml( PARAM, file="PARAM_valid.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Oct 12 04:22:47 2025"

# FINAL

Final Training Dataset


In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/competencia_01.csv.gz", stringsAsFactors= TRUE)

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

dataset[,clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L) ]
#dataset_train[,clase01 := ifelse(clase_ternaria %in% c("BAJA+2"), 1L, 0L)]

In [ ]:
dataset_train_final <- dataset[foto_mes %in% PARAM$train_final]
dataset_train_final[,.N,clase_ternaria]

clase_ternaria,N
<fct>,<int>
CONTINUA,642824
BAJA+2,3938
BAJA+1,3447


In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain_final <- lgb.Dataset(
  data= data.matrix(dataset_train_final[, campos_buenos, with= FALSE]),
  label= dataset_train_final[, clase01]
)

In [ ]:
param_final <- modifyList(PARAM$lgbm$param_fijos, PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 102191

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 6.771127

$lambda_l2
[1] 5.828662

$max_bin
[1] 31

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 994

$learning_rate
[1] 0.04483601

$feature_fraction
[1] 0.1651085

$num_leaves
[1] 2047

$min_data_in_leaf
[1] 832

In [ ]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)
param_normalizado$bagging_fraction <- 0.9
param_normalizado$bagging_freq     <- 1L


In [ ]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes %in% PARAM$future]

In [ ]:
# --- ensemble multi-seed ---
SEEDS <- c(102191, 230101, 999199, 878787, 761177, 161803, 550051, 420420, 314159, 271828)

preds_list <- vector("list", length(SEEDS))
impo_list  <- vector("list", length(SEEDS))

for (i in seq_along(SEEDS)) {
  si <- SEEDS[i]
  p  <- modifyList(param_normalizado, list(seed = si, bagging_seed = si, feature_fraction_seed = si))
  modelo_i <- lgb.train(data = dtrain_final, param = p)

  # guardo importancia por seed (luego promedio)
  impo_list[[i]] <- as.data.table(lgb.importance(modelo_i))[ , seed := si ]

  # guardo el modelo de cada seed
  lgb.save(modelo_i, sprintf("modelo_seed_%d.txt", si))

  # predicción de esta seed
  preds_list[[i]] <- predict(modelo_i, data.matrix(dfuture[, campos_buenos, with = FALSE]))
  rm(modelo_i); gc()
}

In [ ]:
# promedio
pred_mat <- do.call(cbind, preds_list)

# tabla de predicción final (promedio de semillas)
tb_prediccion <- dfuture[, .(numero_de_cliente, foto_mes)]
tb_prediccion[, prob := rowMeans(pred_mat)]   # o: apply(pred_mat, 1, median)

In [ ]:
# Importancia PROMEDIO del ensemble
tb_importancia <- rbindlist(impo_list, fill = TRUE)[
  , .(
      gain_mean   = mean(Gain, na.rm = TRUE),
      gain_median = median(Gain, na.rm = TRUE),
      freq_mean   = mean(Frequency, na.rm = TRUE)
    ),
  by = Feature
][order(-gain_mean)]

fwrite(tb_importancia, file = "impo_ensemble.txt", sep = "\t")
fwrite(tb_prediccion,  file = "prediccion.txt",     sep = "\t")

Aplico el modelo final a los datos del futuro

In [ ]:
# inicilizo el dataset  drealidad
drealidad <- realidad_inicializar( dfuture, PARAM)

Kaggle Competition Submit

In [ ]:
PARAM$cortes

[1]  6000  6250  6500  6750  7000  7250  7500  7750  8000  8250  8500  8750
[13]  9000  9250  9500  9750 10000 10250 10500 10750 11000 11250 11500 11750
[25] 12000 12250 12500 12750 13000 13250 13500 13750 14000 14250 14500 14750
[37] 15000 15250 15500 15750 16000 16250 16500 16750 17000 17250 17500 17750
[49] 18000 18250 18500 18750 19000

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marco los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  res <- realidad_evaluar( drealidad, tb_prediccion)

  options(scipen = 999)
  cat( "Envios=", envios, "\t",
    " TOTAL=", res$total,
    "  Public=", res$public,
    " Private=", res$private,
    "\n",
    sep= ""
  )

}

Envios=6000	 TOTAL=-120000000  Public=-122066667 Private=-119114286
Envios=6250	 TOTAL=-125000000  Public=-127333333 Private=-124000000
Envios=6500	 TOTAL=-130000000  Public=-131400000 Private=-129400000
Envios=6750	 TOTAL=-135000000  Public=-136066667 Private=-134542857
Envios=7000	 TOTAL=-140000000  Public=-141200000 Private=-139485714
Envios=7250	 TOTAL=-145000000  Public=-145866667 Private=-144628571
Envios=7500	 TOTAL=-150000000  Public=-151933333 Private=-149171429
Envios=7750	 TOTAL=-155000000  Public=-156933333 Private=-154171429
Envios=8000	 TOTAL=-160000000  Public=-162666667 Private=-158857143
Envios=8250	 TOTAL=-165000000  Public=-167066667 Private=-164114286
Envios=8500	 TOTAL=-170000000  Public=-171933333 Private=-169171429
Envios=8750	 TOTAL=-175000000  Public=-176666667 Private=-174285714
Envios=9000	 TOTAL=-180000000  Public=-181400000 Private=-179400000
Envios=9250	 TOTAL=-185000000  Public=-186200000 Private=-184485714
Envios=9500	 TOTAL=-190000000  Public=-191466667

In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Oct 12 05:18:20 2025"